<a href="https://colab.research.google.com/github/un1u3/ml-labs/blob/main/Deep%20Learning/Pytorch/Data_%26_Data_Loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np

In [17]:
class CSVDataset(Dataset):

  def __init__(self, csv_file, target_col):
    df = pd.read_csv(csv_file)
    self.x = df.drop(columns=[target_col],axis=1).values
    self.y = df[target_col].values

    self.x = torch.FloatTensor(self.x)
    self.y = torch.LongTensor(self.y)
    print(f"✅ Loaded {len(self.x)} samples with {self.x.shape[1]} features")

  def __len__(self):
      return len(self.x)

  def __getitem__(self, idx):
      return self.x[idx], self.y[idx]

In [18]:
# Create a sample CSV file first
df = pd.DataFrame({
    'feature1': np.random.randn(200),
    'feature2': np.random.randn(200),
    'feature3': np.random.randn(200),
    'target': np.random.randint(0, 2, 200)  # Binary: 0 or 1
})
df.to_csv('data.csv', index=False)
print("✅ Created sample data.csv")

# Now use our dataset class
dataset = CSVDataset('data.csv', target_col='target')

# Check what we got
print(f"\n📊 Dataset Info:")
print(f"Total samples: {len(dataset)}")
print(f"\n🔍 First sample:")
print(f"Features: {dataset[0][0]}")
print(f"Label: {dataset[0][1]}")
print(f"\n📏 Feature shape: {dataset[0][0].shape}")

✅ Created sample data.csv
✅ Loaded 200 samples with 3 features

📊 Dataset Info:
Total samples: 200

🔍 First sample:
Features: tensor([-1.0809, -0.2910, -0.5743])
Label: 1

📏 Feature shape: torch.Size([3])


# DataLoader
- dividing data into chunks so that we can train efficiently

In [27]:
from torch.utils.data import DataLoader
import torch.nn as nn

In [28]:
loader  = DataLoader(
    dataset,
    batch_size=32, # processes 32 samples together
    shuffle = True, # mix up data each time
    drop_last= False # keeps the incomplete batch
)
print(f"📦 Total batches: {len(loader)}")
print(f"   (200 samples ÷ 32 per batch = {200/32:.1f} batches)")


📦 Total batches: 7
   (200 samples ÷ 32 per batch = 6.2 batches)


In [29]:
for feature, labels in loader:
  print(feature.shape)
  print(labels.shape)

torch.Size([32, 3])
torch.Size([32])
torch.Size([32, 3])
torch.Size([32])
torch.Size([32, 3])
torch.Size([32])
torch.Size([32, 3])
torch.Size([32])
torch.Size([32, 3])
torch.Size([32])
torch.Size([32, 3])
torch.Size([32])
torch.Size([8, 3])
torch.Size([8])


1. **`batch_size=32`**
    
    - Process 32 samples at once
    - Smaller = more updates, slower training
    - Larger = fewer updates, needs more memory
    - 32 is a good starting point!
2. **`shuffle=True`**
    
    - Randomize order every epoch
    - Helps model generalize better
    - Always True for training!
3. **`drop_last=False`**
    
    - What if 200 samples ÷ 32 = 6.25 batches?
    - `False` = Keep that last batch of 8 samples
    - `True` = Drop it

In [31]:
# using dataloader in training
import torch.nn as nn
# simple model
model = nn.Sequential(
    nn.Linear(3,64),
    nn.ReLU(),
    nn.Linear(64,2),
)

for epochs in range(3):
  for batch_idx, (features, labels) in enumerate(loader):
    outputs = model(features)
    if batch_idx == 0:  # First batch only
            print(f"   Batch {batch_idx + 1}: {features.shape}")

   Batch 1: torch.Size([32, 3])
   Batch 1: torch.Size([32, 3])
   Batch 1: torch.Size([32, 3])
